# 🏛️ KG Hà Nội — Bước 4: KGRetriever Module

**Input:** `kg/graph.pkl` (từ bước 3)
**Output:** `kg_retriever.py` — module import được, sẵn sàng plug vào VLM

**Class KGRetriever:**
- `keyword_match()` — tìm entity bằng tên trong câu hỏi
- `visual_match()` — tìm entity bằng I-JEPA embedding (khi có gallery)
- `retrieve()` — PPR + reasoning paths → text context
- `query()` — full pipeline: seed selection → retrieve → textualize

**Bài báo tham khảo:**
- HippoRAG — Personalized PageRank từ seed nodes
- RoG — Reasoning paths thay vì flat subgraph
- G-Retriever — Text embedding indexing

## 0. Setup & Load Graph

In [ ]:
!pip install sentence-transformers networkx -q

import json, re, pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer

# ---- ĐƯỜNG DẪN ----
GRAPH_PATH = "/kaggle/working/kg/graph.pkl"

# Nếu graph nằm trong input dataset
if not Path(GRAPH_PATH).exists():
    alt = "/kaggle/input/kg-graph/kg/graph.pkl"
    if Path(alt).exists():
        GRAPH_PATH = alt

# Load
with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Path: {GRAPH_PATH}")

# Load encoder
encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Encoder loaded ✓")

## 1. KGRetriever Class

In [ ]:
class KGRetriever:
    """
    Knowledge Graph Retriever cho di tích Hà Nội.

    Pipeline:
      1. Seed selection: keyword match → embedding fallback → visual match
      2. Personalized PageRank từ seed (HippoRAG)
      3. Extract reasoning paths (RoG)
      4. Textualize → context string cho LLM

    Usage:
      retriever = KGRetriever("kg/graph.pkl")
      entity, confidence, context = retriever.query(question="Văn Miếu xây năm nào?")
      # hoặc với ảnh:
      entity, confidence, context = retriever.query(image_embedding=img_emb, question="...")
    """

    def __init__(self, graph_path: str, encoder_name: str = "paraphrase-multilingual-MiniLM-L12-v2"):
        # Load graph
        with open(graph_path, "rb") as f:
            self.G = pickle.load(f)

        # Encoder (lazy load nếu cần)
        self._encoder = None
        self._encoder_name = encoder_name

        # Pre-index
        self._landmarks = [n for n in self.G if self.G.nodes[n].get("type") == "Landmark"]
        self._restaurants = [n for n in self.G if self.G.nodes[n].get("type") == "Restaurant"]
        self._priority_nodes = self._landmarks + self._restaurants

        # Pre-convert image embeddings
        self._img_embs = {}
        for nid in self.G.nodes:
            if "image_embedding" in self.G.nodes[nid]:
                self._img_embs[nid] = np.array(self.G.nodes[nid]["image_embedding"])

        # Undirected version (for PPR + path finding)
        self._G_und = self.G.to_undirected()

        print(f"KGRetriever: {self.G.number_of_nodes()} nodes, "
              f"{len(self._landmarks)} landmarks, "
              f"{len(self._restaurants)} restaurants")

    @property
    def encoder(self):
        if self._encoder is None:
            self._encoder = SentenceTransformer(self._encoder_name)
        return self._encoder

    # ================================================================
    # SEED SELECTION
    # ================================================================

    def keyword_match(self, question: str) -> tuple:
        """
        Tìm entity bằng keyword matching trong câu hỏi.
        Returns: (node_id, match_length) hoặc (None, 0)
        """
        question_lower = question.lower()
        best_node, best_len = None, 0

        for nid in self._priority_nodes:
            nid_lower = nid.lower()

            # Check 1: tên node nguyên vẹn
            if nid_lower in question_lower and len(nid) > best_len:
                best_node, best_len = nid, len(nid)

            # Check 2: tách theo dấu gạch
            # "Văn Miếu – Quốc Tử Giám" → ["Văn Miếu", "Quốc Tử Giám"]
            name_parts = re.split(r'\s*[–\-]\s*', nid)
            for part in name_parts:
                part = part.strip()
                if len(part) >= 3 and part.lower() in question_lower and len(part) > best_len:
                    best_node, best_len = nid, len(part)

            # Check 3: aliases
            for alias in self.G.nodes[nid].get("aliases", []):
                if len(alias) >= 3 and alias.lower() in question_lower and len(alias) > best_len:
                    best_node, best_len = nid, len(alias)

        return best_node, best_len

    def embedding_match(self, question: str, top_k: int = 1) -> list:
        """
        Tìm entity bằng cosine similarity (fallback).
        Returns: [(node_id, score), ...]
        """
        q_emb = self.encoder.encode(question)
        scores = {}

        for nid in self._priority_nodes:
            if "text_embedding" in self.G.nodes[nid]:
                emb = np.array(self.G.nodes[nid]["text_embedding"])
                cos = float(np.dot(q_emb, emb) / (
                    np.linalg.norm(q_emb) * np.linalg.norm(emb) + 1e-8))
                scores[nid] = cos

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def visual_match(self, image_embedding: np.ndarray, top_k: int = 3) -> list:
        """
        Tìm entity bằng I-JEPA image embedding.
        Returns: [(node_id, score), ...]
        """
        if not self._img_embs:
            return []

        scores = {}
        for nid, emb in self._img_embs.items():
            cos = float(np.dot(image_embedding, emb) / (
                np.linalg.norm(image_embedding) * np.linalg.norm(emb) + 1e-8))
            scores[nid] = cos

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def find_seed(self, question: str = None, image_embedding: np.ndarray = None,
                  confidence_threshold: float = 0.3) -> tuple:
        """
        Tìm seed entity theo thứ tự ưu tiên:
          1. Keyword match (chính xác nhất)
          2. Visual match (nếu có ảnh)
          3. Embedding match (fallback)

        Returns: (node_id, confidence, method)
        """
        # 1) Keyword match
        if question:
            node, match_len = self.keyword_match(question)
            if node:
                confidence = min(match_len / 10.0, 1.0)  # normalize
                return node, confidence, "keyword"

        # 2) Visual match
        if image_embedding is not None:
            results = self.visual_match(image_embedding)
            if results and results[0][1] >= confidence_threshold:
                return results[0][0], results[0][1], "visual"

        # 3) Embedding match
        if question:
            results = self.embedding_match(question)
            if results:
                return results[0][0], results[0][1], "embedding"

        return None, 0.0, "none"

    # ================================================================
    # RETRIEVAL (PPR + Paths)
    # ================================================================

    def personalized_pagerank(self, seeds: list, alpha: float = 0.15,
                               top_n: int = 20) -> list:
        """
        Personalized PageRank từ seed nodes (HippoRAG).
        Returns: [(node_id, score), ...] sorted by score desc
        """
        valid_seeds = [s for s in seeds if s in self._G_und]
        if not valid_seeds:
            return []

        personalization = {}
        for s in valid_seeds:
            personalization[s] = 1.0 / len(valid_seeds)

        try:
            ppr = nx.pagerank(self._G_und, alpha=alpha,
                              personalization=personalization, max_iter=100)
        except nx.PowerIterationFailedConvergence:
            ppr = personalization

        ranked = sorted(ppr.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_n]

    def extract_paths(self, seed: str, targets: list,
                      max_hops: int = 2) -> list:
        """
        Extract reasoning paths từ seed đến targets (RoG style).
        Returns: list of (src, relation, dst) tuples
        """
        triples = []
        seen = set()

        for target in targets:
            if target == seed:
                continue
            try:
                for path in nx.all_simple_paths(self._G_und, seed, target,
                                                 cutoff=max_hops):
                    for i in range(len(path) - 1):
                        s, d = path[i], path[i + 1]
                        # Try both directions
                        edge = self.G.get_edge_data(s, d)
                        if edge:
                            rel = edge.get("relation", "liên_quan")
                            src, dst = s, d
                        else:
                            edge = self.G.get_edge_data(d, s)
                            rel = edge.get("relation", "liên_quan") if edge else "liên_quan"
                            src, dst = d, s

                        key = (src, rel, dst)
                        if key not in seen:
                            triples.append((src, rel, dst))
                            seen.add(key)
            except (nx.NodeNotFound, nx.NetworkXError):
                continue

        return triples

    def textualize(self, entity: str, triples: list, max_lines: int = 15) -> str:
        """
        Chuyển entity + triples thành context text cho LLM.
        """
        lines = []

        # Entity description
        desc = self.G.nodes.get(entity, {})
        entity_type = desc.get("type", "")
        if entity_type:
            lines.append(f"[{entity_type}] {entity}")

        # Key attributes từ frontmatter
        for key in ("xây_dựng_năm", "thuộc_quận", "triều_đại",
                     "phong_cách_kiến_trúc", "loại_món", "tôn_giáo"):
            val = desc.get(key)
            if val:
                lines.append(f"  {key}: {val}")

        lines.append("")

        # Reasoning paths
        for src, rel, dst in triples[:max_lines]:
            lines.append(f"  {src} → [{rel}] → {dst}")

        return "\n".join(lines)

    # ================================================================
    # FULL QUERY PIPELINE
    # ================================================================

    def query(self, question: str = None, image_embedding: np.ndarray = None,
              top_n_ppr: int = 15, max_hops: int = 2,
              confidence_threshold: float = 0.3) -> dict:
        """
        Full retrieval pipeline.

        Args:
            question: câu hỏi text
            image_embedding: I-JEPA embedding của ảnh (optional)
            top_n_ppr: số nodes lấy từ PPR
            max_hops: max hops cho reasoning paths
            confidence_threshold: ngưỡng confidence cho visual match

        Returns: dict với keys:
            entity: tên entity tìm được
            confidence: confidence score
            method: "keyword" | "visual" | "embedding" | "none"
            context: text context cho LLM
            triples: list of (src, rel, dst)
            ppr_scores: top PPR scores
        """
        # 1) Find seed
        entity, confidence, method = self.find_seed(
            question=question,
            image_embedding=image_embedding,
            confidence_threshold=confidence_threshold
        )

        if entity is None:
            return {
                "entity": None, "confidence": 0.0, "method": "none",
                "context": "Không nhận diện được di tích.",
                "triples": [], "ppr_scores": [],
            }

        # 2) PPR
        ppr_results = self.personalized_pagerank([entity], top_n=top_n_ppr)
        ppr_nodes = [n for n, _ in ppr_results]

        # 3) Extract paths
        triples = self.extract_paths(entity, ppr_nodes, max_hops=max_hops)

        # 4) Textualize
        context = self.textualize(entity, triples)

        return {
            "entity": entity,
            "confidence": confidence,
            "method": method,
            "context": context,
            "triples": triples,
            "ppr_scores": ppr_results[:10],
        }

    # ================================================================
    # UTILITIES
    # ================================================================

    def get_entity_info(self, entity: str) -> dict:
        """Lấy toàn bộ thông tin 1 entity."""
        if entity not in self.G:
            return {}
        attrs = dict(self.G.nodes[entity])
        attrs.pop("text_embedding", None)
        attrs.pop("image_embedding", None)

        edges_out = [(entity, d, self.G.edges[entity, d].get("relation", "?"))
                     for d in self.G.successors(entity)]
        edges_in = [(s, entity, self.G.edges[s, entity].get("relation", "?"))
                    for s in self.G.predecessors(entity)]

        return {"attrs": attrs, "edges_out": edges_out, "edges_in": edges_in}

    def list_landmarks(self) -> list:
        return sorted(self._landmarks)

    def list_restaurants(self) -> list:
        return sorted(self._restaurants)

    def stats(self):
        print(f"Nodes: {self.G.number_of_nodes()}")
        print(f"Edges: {self.G.number_of_edges()}")
        print(f"Landmarks: {len(self._landmarks)}")
        print(f"Restaurants: {len(self._restaurants)}")
        print(f"Image embeddings: {len(self._img_embs)}")
        components = list(nx.connected_components(self._G_und))
        print(f"Connected components: {len(components)}")


print("✓ KGRetriever class defined")

## 2. Test KGRetriever

In [ ]:
# ---- Khởi tạo ----
retriever = KGRetriever(GRAPH_PATH)
retriever.stats()

In [ ]:
# ---- Test queries ----
test_questions = [
    "Văn Miếu xây dựng năm nào?",
    "Chùa Một Cột ở quận nào?",
    "Quán phở nào gần Hồ Hoàn Kiếm?",
    "Nhà hát Lớn Hà Nội do ai thiết kế?",
    "Cầu Long Biên xây năm nào?",
    "Chùa Trấn Quốc thuộc triều đại nào?",
    "Nhà tù Hỏa Lò có tên khác là gì?",
    "Đền Ngọc Sơn thờ ai?",
]

for q in test_questions:
    print(f"\n{'='*55}")
    print(f"Q: {q}")
    print(f"{'='*55}")
    result = retriever.query(question=q)
    print(f"  Entity: {result['entity']}  ({result['method']}, conf={result['confidence']:.2f})")
    print(f"  Context:")
    print(result['context'])

## 3. Test chi tiết — Entity Info

In [ ]:
# ---- Xem chi tiết 1 entity ----
test_entity = "Chùa Một Cột"  # ← đổi tùy ý

# Tìm entity trong graph
found = None
for n in retriever.list_landmarks():
    if test_entity.lower() in n.lower():
        found = n
        break

if found:
    info = retriever.get_entity_info(found)
    print(f"Entity: {found}")
    print(f"\nAttributes:")
    for k, v in info['attrs'].items():
        if k not in ('_file',):
            print(f"  {k}: {v}")
    print(f"\nEdges out ({len(info['edges_out'])}):")
    for src, dst, rel in info['edges_out']:
        print(f"  → [{rel}] → {dst}")
    print(f"\nEdges in ({len(info['edges_in'])}):")
    for src, dst, rel in info['edges_in']:
        print(f"  {src} → [{rel}] →")
else:
    print(f"Không tìm thấy: {test_entity}")
    print(f"\nLandmarks có:")
    for l in retriever.list_landmarks()[:20]:
        print(f"  {l}")

## 4. Test Late Fusion Prompt

Mô phỏng cách KG context sẽ được merge với VLM output trong pipeline hoàn chỉnh.

In [ ]:
def simulate_late_fusion(question, vlm_answer, kg_context):
    """Mô phỏng prompt merge cho Qwen."""
    prompt = f"""Dựa trên phân tích hình ảnh và thông tin lịch sử dưới đây,
hãy trả lời câu hỏi của du khách bằng tiếng Việt.

[Phân tích hình ảnh]
{vlm_answer}

[Thông tin lịch sử]
{kg_context}

[Câu hỏi]
{question}

Hướng dẫn:
- Ưu tiên thông tin lịch sử cho năm tháng, tên gọi, triều đại
- Sử dụng phân tích hình ảnh cho mô tả kiến trúc, tình trạng hiện tại
- Trả lời ngắn gọn, chính xác"""
    return prompt


# ---- Simulate ----
question = "Di tích này xây dựng năm nào và có đặc điểm gì?"

# Giả lập VLM output
vlm_answer = "Đây là một ngôi đền cổ với kiến trúc truyền thống Việt Nam, có cổng tam quan, mái ngói cong, và hồ nước phía trước."

# KG retrieval
result = retriever.query(question=question)
kg_context = result['context']

# Build prompt
prompt = simulate_late_fusion(question, vlm_answer, kg_context)

print("=" * 55)
print("LATE FUSION PROMPT")
print("=" * 55)
print(prompt)
print("\n" + "=" * 55)
print(f"Entity: {result['entity']}")
print(f"Method: {result['method']}")
print(f"Prompt length: {len(prompt)} chars, ~{len(prompt)//4} tokens")

## 5. Export Module

In [ ]:
# ---- Lưu class thành file .py import được ----
module_code = '''"""
KGRetriever — Knowledge Graph Retriever cho di tích Hà Nội.

Usage:
    from kg_retriever import KGRetriever

    retriever = KGRetriever("kg/graph.pkl")
    result = retriever.query(question="Văn Miếu xây năm nào?")
    print(result["context"])
"""

import re, pickle
from collections import defaultdict

import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer


class KGRetriever:

    def __init__(self, graph_path, encoder_name="paraphrase-multilingual-MiniLM-L12-v2"):
        with open(graph_path, "rb") as f:
            self.G = pickle.load(f)
        self._encoder = None
        self._encoder_name = encoder_name
        self._landmarks = [n for n in self.G if self.G.nodes[n].get("type") == "Landmark"]
        self._restaurants = [n for n in self.G if self.G.nodes[n].get("type") == "Restaurant"]
        self._priority_nodes = self._landmarks + self._restaurants
        self._img_embs = {}
        for nid in self.G.nodes:
            if "image_embedding" in self.G.nodes[nid]:
                self._img_embs[nid] = np.array(self.G.nodes[nid]["image_embedding"])
        self._G_und = self.G.to_undirected()

    @property
    def encoder(self):
        if self._encoder is None:
            self._encoder = SentenceTransformer(self._encoder_name)
        return self._encoder

    def keyword_match(self, question):
        question_lower = question.lower()
        best_node, best_len = None, 0
        for nid in self._priority_nodes:
            nid_lower = nid.lower()
            if nid_lower in question_lower and len(nid) > best_len:
                best_node, best_len = nid, len(nid)
            name_parts = re.split(r"\\s*[–\\-]\\s*", nid)
            for part in name_parts:
                part = part.strip()
                if len(part) >= 3 and part.lower() in question_lower and len(part) > best_len:
                    best_node, best_len = nid, len(part)
            for alias in self.G.nodes[nid].get("aliases", []):
                if len(alias) >= 3 and alias.lower() in question_lower and len(alias) > best_len:
                    best_node, best_len = nid, len(alias)
        return best_node, best_len

    def embedding_match(self, question, top_k=1):
        q_emb = self.encoder.encode(question)
        scores = {}
        for nid in self._priority_nodes:
            if "text_embedding" in self.G.nodes[nid]:
                emb = np.array(self.G.nodes[nid]["text_embedding"])
                cos = float(np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb) + 1e-8))
                scores[nid] = cos
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def visual_match(self, image_embedding, top_k=3):
        if not self._img_embs:
            return []
        scores = {}
        for nid, emb in self._img_embs.items():
            cos = float(np.dot(image_embedding, emb) / (np.linalg.norm(image_embedding) * np.linalg.norm(emb) + 1e-8))
            scores[nid] = cos
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def find_seed(self, question=None, image_embedding=None, confidence_threshold=0.3):
        if question:
            node, match_len = self.keyword_match(question)
            if node:
                return node, min(match_len / 10.0, 1.0), "keyword"
        if image_embedding is not None:
            results = self.visual_match(image_embedding)
            if results and results[0][1] >= confidence_threshold:
                return results[0][0], results[0][1], "visual"
        if question:
            results = self.embedding_match(question)
            if results:
                return results[0][0], results[0][1], "embedding"
        return None, 0.0, "none"

    def personalized_pagerank(self, seeds, alpha=0.15, top_n=20):
        valid = [s for s in seeds if s in self._G_und]
        if not valid:
            return []
        pers = {s: 1.0 / len(valid) for s in valid}
        try:
            ppr = nx.pagerank(self._G_und, alpha=alpha, personalization=pers, max_iter=100)
        except:
            ppr = pers
        return sorted(ppr.items(), key=lambda x: x[1], reverse=True)[:top_n]

    def extract_paths(self, seed, targets, max_hops=2):
        triples, seen = [], set()
        for target in targets:
            if target == seed:
                continue
            try:
                for path in nx.all_simple_paths(self._G_und, seed, target, cutoff=max_hops):
                    for i in range(len(path) - 1):
                        s, d = path[i], path[i + 1]
                        edge = self.G.get_edge_data(s, d)
                        if edge:
                            rel, src, dst = edge.get("relation", "liên_quan"), s, d
                        else:
                            edge = self.G.get_edge_data(d, s)
                            rel = edge.get("relation", "liên_quan") if edge else "liên_quan"
                            src, dst = d, s
                        key = (src, rel, dst)
                        if key not in seen:
                            triples.append((src, rel, dst))
                            seen.add(key)
            except:
                continue
        return triples

    def textualize(self, entity, triples, max_lines=15):
        lines = []
        desc = self.G.nodes.get(entity, {})
        etype = desc.get("type", "")
        if etype:
            lines.append(f"[{etype}] {entity}")
        for key in ("xây_dựng_năm", "thuộc_quận", "triều_đại", "phong_cách_kiến_trúc", "loại_món", "tôn_giáo"):
            val = desc.get(key)
            if val:
                lines.append(f"  {key}: {val}")
        lines.append("")
        for src, rel, dst in triples[:max_lines]:
            lines.append(f"  {src} → [{rel}] → {dst}")
        return "\n".join(lines)

    def query(self, question=None, image_embedding=None, top_n_ppr=15, max_hops=2, confidence_threshold=0.3):
        entity, confidence, method = self.find_seed(question=question, image_embedding=image_embedding, confidence_threshold=confidence_threshold)
        if entity is None:
            return {"entity": None, "confidence": 0.0, "method": "none", "context": "Không nhận diện được di tích.", "triples": [], "ppr_scores": []}
        ppr_results = self.personalized_pagerank([entity], top_n=top_n_ppr)
        ppr_nodes = [n for n, _ in ppr_results]
        triples = self.extract_paths(entity, ppr_nodes, max_hops=max_hops)
        context = self.textualize(entity, triples)
        return {"entity": entity, "confidence": confidence, "method": method, "context": context, "triples": triples, "ppr_scores": ppr_results[:10]}

    def get_entity_info(self, entity):
        if entity not in self.G:
            return {}
        attrs = dict(self.G.nodes[entity])
        attrs.pop("text_embedding", None)
        attrs.pop("image_embedding", None)
        edges_out = [(entity, d, self.G.edges[entity, d].get("relation", "?")) for d in self.G.successors(entity)]
        edges_in = [(s, entity, self.G.edges[s, entity].get("relation", "?")) for s in self.G.predecessors(entity)]
        return {"attrs": attrs, "edges_out": edges_out, "edges_in": edges_in}

    def list_landmarks(self):
        return sorted(self._landmarks)

    def list_restaurants(self):
        return sorted(self._restaurants)

    def stats(self):
        print(f"Nodes: {self.G.number_of_nodes()}, Edges: {self.G.number_of_edges()}")
        print(f"Landmarks: {len(self._landmarks)}, Restaurants: {len(self._restaurants)}")
        print(f"Image embeddings: {len(self._img_embs)}")
'''

# Fix regex escaping cho file .py
module_code = module_code.replace('\\\\s*', '\\s*').replace('\\\\-', '\\-')

out_path = Path("/kaggle/working/kg_retriever.py")
out_path.write_text(module_code, encoding="utf-8")
print(f"✓ Saved: {out_path}")
print(f"\nImport từ pipeline khác:")
print(f"  from kg_retriever import KGRetriever")
print(f"  retriever = KGRetriever('kg/graph.pkl')")

## 6. Download

In [ ]:
import shutil

# Copy module + graph vào 1 thư mục
export_dir = Path("/kaggle/working/kg_export")
export_dir.mkdir(exist_ok=True)
shutil.copy("/kaggle/working/kg_retriever.py", export_dir / "kg_retriever.py")
if Path("/kaggle/working/kg/graph.pkl").exists():
    shutil.copy("/kaggle/working/kg/graph.pkl", export_dir / "graph.pkl")
if Path("/kaggle/working/kg/meta.json").exists():
    shutil.copy("/kaggle/working/kg/meta.json", export_dir / "meta.json")

shutil.make_archive("/kaggle/working/kg_module", "zip", str(export_dir))
print(f"✓ /kaggle/working/kg_module.zip")
print(f"\nChứa:")
for f in sorted(export_dir.iterdir()):
    size = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size:.1f} MB")